# Real-World Data Analytics and Machine Learning
## Revised, fully guided notebook

This version of the notebook is written to be easier to follow, easier to run, and easier to adapt for a new dataset.

### What has been improved
- automatic package checking and installation
- clearer explanations before every code section
- guided data upload / file selection
- more comments inside the code
- better structure for data cleaning, exploration, PCA, and classification
- cleaner model comparison tables and interpretation outputs

### Expected data assumptions
This notebook assumes:
- the dataset contains a target column called **`Diagnosis`**
- an identifier column called **`ID`** may exist and should be removed before modelling
- the data file is supplied by the user at runtime as `.xlsx`, `.xls`, or `.csv`


## Step 1 — Check and install the required Python packages

This cell checks whether the notebook environment already contains the required libraries.  
If any package is missing, it will be installed automatically using `pip`.

This makes the notebook easier to run on:
- Jupyter Notebook
- JupyterLab
- VS Code notebooks
- Google Colab


In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "ipywidgets": "ipywidgets"
}

missing = []

for import_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(import_name)
        print(f"✓ {package_name} is already installed")
    except ImportError:
        print(f"✗ {package_name} is missing and will be installed")
        missing.append(package_name)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("\nAll missing packages have now been installed.")
else:
    print("\nAll required packages are already available.")


## Step 2 — Import libraries and configure the notebook environment

This cell imports all libraries used in the analysis and sets a few display options so that outputs are easier to read.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")


## Step 3 — Upload or locate the dataset file

This cell helps the user provide the dataset file.

### Supported file formats
- `.xlsx`
- `.xls`
- `.csv`

### How it works
- In **Google Colab**, the file upload window will open automatically.
- In **Jupyter / VS Code**, you can type the full file path when prompted.

After the file is provided, the notebook reads the dataset into a pandas DataFrame named **`df`**.


In [ ]:
def load_dataset_interactively():
    file_path = None

    # Try Google Colab upload first
    try:
        from google.colab import files  # type: ignore
        print("Google Colab environment detected.")
        uploaded = files.upload()
        if uploaded:
            file_path = next(iter(uploaded.keys()))
            print(f"Uploaded file detected: {file_path}")
    except Exception:
        pass

    # Fallback: ask the user for a local file path
    if file_path is None:
        file_path = input(
            "Enter the full path to your data file (.xlsx, .xls, or .csv): "
        ).strip()

    if not file_path:
        raise ValueError("No file path was provided.")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = os.path.splitext(file_path)[1].lower()

    if extension in [".xlsx", ".xls"]:
        df_local = pd.read_excel(file_path)
    elif extension == ".csv":
        df_local = pd.read_csv(file_path)
    else:
        raise ValueError("Unsupported file format. Please use .xlsx, .xls, or .csv")

    return df_local, file_path

df, DATA_FILE = load_dataset_interactively()

print(f"Dataset loaded successfully from: {DATA_FILE}")
print(f"Shape of the dataset: {df.shape}")
display(df.head())


## Step 4 — Inspect the dataset structure and missing values

This section gives a quick overview of the dataset:
- number of rows and columns
- column names and data types
- first few records
- missing value counts

This helps us verify that the dataset has loaded correctly before any cleaning or modelling starts.


In [ ]:
print("Dataset information:")
display(df.info())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values per column:")
missing_data = df.isnull().sum().sort_values(ascending=False)
display(missing_data[missing_data > 0])

if missing_data.sum() == 0:
    print("No missing values were detected in the dataset.")


## Step 5 — Remove non-informative identifier columns

Identifier fields such as `ID` are usually not predictive.  
If an `ID` column exists, it is removed so it does not affect the analysis or model training.


In [ ]:
if "ID" in df.columns:
    df = df.drop(columns=["ID"])
    print("The 'ID' column was found and removed.")
else:
    print("No 'ID' column was found, so nothing was removed.")

print(f"Updated dataset shape: {df.shape}")
display(df.head())


## Step 6 — Check the target variable distribution

This section visualises the class balance of the target variable **`Diagnosis`**.

Understanding class balance is important because:
- a strongly imbalanced target can bias model performance
- accuracy alone may become misleading
- we may later need stratified splitting or class-aware metrics


In [ ]:
if "Diagnosis" not in df.columns:
    raise KeyError("The dataset must contain a target column called 'Diagnosis'.")

diagnosis_counts = df["Diagnosis"].value_counts()

print("Diagnosis class counts:")
display(diagnosis_counts)

plt.figure(figsize=(6, 6))
plt.pie(
    diagnosis_counts,
    labels=diagnosis_counts.index,
    autopct="%1.1f%%",
    startangle=140
)
plt.title("Distribution of Diagnosis")
plt.axis("equal")
plt.show()


## Step 7 — Generate summary statistics for numerical variables

This section provides descriptive statistics for all numeric columns, including:
- count
- mean
- standard deviation
- minimum
- quartiles
- maximum

These values help us understand the scale, spread, and possible skewness of the variables.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if not numeric_cols:
    raise ValueError("No numeric columns were found in the dataset.")

numeric_summary = df[numeric_cols].describe().T
print("Summary statistics for numeric variables:")
display(numeric_summary)


## Step 8 — Visual inspection of outliers using boxplots

Boxplots help us quickly identify:
- extreme values
- possible outliers
- skewed distributions
- variables with very different scales

These plots show the numeric variables before outlier removal.


In [ ]:
n_cols = 5
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_ylabel(col)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Step 9 — Detect possible outliers using the Z-score method

This cell calculates the absolute Z-score for each numeric feature and flags rows where at least one variable has a Z-score above the chosen threshold.

### Current threshold
- `3.0`

This is a common rule of thumb for identifying extreme observations.


In [ ]:
z_score_threshold = 3.0

z_scores = np.abs(stats.zscore(df[numeric_cols], nan_policy="omit"))
outlier_mask = (z_scores > z_score_threshold).any(axis=1)

outlier_rows = df.loc[outlier_mask].copy()

print(f"Number of rows identified as potential outliers: {outlier_mask.sum()}")
display(outlier_rows.head(20))


## Step 10 — Remove the detected outliers and create a cleaned dataset

Outlier removal is performed here only for rows flagged in the previous step.

The cleaned DataFrame is stored in **`df_clean`** so that:
- the original data remains available if needed
- the rest of the analysis can continue using a cleaner dataset


In [ ]:
df_clean = df.loc[~outlier_mask].reset_index(drop=True)

print(f"Original dataset shape: {df.shape}")
print(f"Cleaned dataset shape : {df_clean.shape}")
print(f"Rows removed          : {df.shape[0] - df_clean.shape[0]}")

df = df_clean.copy()
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df.head())


## Step 11 — Re-check boxplots after outlier removal

These boxplots help us verify whether the data distribution looks more stable after removing extreme observations.


In [ ]:
n_cols = 5
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(f"Post-cleaning boxplot of {col}")
    axes[i].set_ylabel(col)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Step 12 — Plot feature distributions

Histograms with KDE curves help us understand:
- central tendency
- spread
- skewness
- whether variables look approximately normal or not

These plots are useful before PCA because PCA is sensitive to variable scaling and structure.


In [ ]:
n_cols = 4
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Step 13 — Create a reduced pairplot for visual relationship checking

A full pairplot can be very large and slow for datasets with many variables.  
To keep the notebook practical, this section selects up to the **6 numeric features with the highest variance** and visualises their pairwise relationships by `Diagnosis`.


In [ ]:
top_pairplot_features = (
    df[numeric_cols].var().sort_values(ascending=False).head(min(6, len(numeric_cols))).index.tolist()
)

print("Variables selected for pairplot:")
print(top_pairplot_features)

sns.pairplot(df[top_pairplot_features + ["Diagnosis"]], hue="Diagnosis", corner=True)
plt.suptitle("Pair Plot of Selected Numeric Variables by Diagnosis", y=1.02)
plt.show()


## Step 14 — Compare distributions by class using violin plots

Violin plots show both:
- the distribution shape
- the class-wise spread

To keep the output readable, this section uses the same reduced feature subset selected for the pairplot.


In [ ]:
selected_violin_features = top_pairplot_features

n_cols = 3
n_rows = math.ceil(len(selected_violin_features) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 5))
axes = axes.flatten()

for i, col in enumerate(selected_violin_features):
    sns.violinplot(x="Diagnosis", y=col, data=df, ax=axes[i])
    axes[i].set_title(f"{col} by Diagnosis")

for j in range(len(selected_violin_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


## Step 15 — Build the correlation heatmap

This heatmap shows the strength and direction of linear relationships among the numeric variables.

It is especially helpful for:
- spotting highly related predictors
- understanding redundancy
- motivating the use of PCA for dimensionality reduction


In [ ]:
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation Matrix of Numeric Variables")
plt.show()


## Step 16 — Standardise numeric features and apply PCA

PCA should be applied on scaled numeric variables so that features with larger magnitudes do not dominate the components.

This section:
1. extracts numeric variables
2. standardises them
3. applies PCA
4. creates a new DataFrame `df_pca`
5. keeps the `Diagnosis` target variable for later modelling


In [ ]:
X_numeric = df[numeric_cols].copy()
y = df["Diagnosis"].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

pca_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pca_cols)
df_pca["Diagnosis"] = y.reset_index(drop=True)

explained_variance_df = pd.DataFrame({
    "Principal Component": pca_cols,
    "Explained Variance Ratio": pca.explained_variance_ratio_,
    "Cumulative Explained Variance": np.cumsum(pca.explained_variance_ratio_)
})

print("Explained variance by principal component:")
display(explained_variance_df.head(15))


## Step 17 — Visualise the explained variance from PCA

These plots help answer two key questions:
- how much variance does each principal component explain?
- how many components are enough to retain most of the information?

The analysis below will use the **first 10 principal components**.


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Explained Variance by Principal Component")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(
    range(1, len(pca.explained_variance_ratio_) + 1),
    np.cumsum(pca.explained_variance_ratio_),
    marker="o"
)
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.axhline(0.95, linestyle="--", label="95% variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.legend()
plt.show()


## Step 18 — Inspect correlations among the first 10 principal components

Principal components should ideally be uncorrelated, so this heatmap acts as a quick validation check.


In [ ]:
num_pcs_to_use = min(10, len(pca_cols))
df_pca_subset = df_pca.iloc[:, :num_pcs_to_use]

correlation_matrix_pca = df_pca_subset.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix_pca, annot=True, cmap="coolwarm", fmt=".2f", annot_kws={"size": 8})
plt.title(f"Correlation Matrix of the First {num_pcs_to_use} Principal Components")
plt.show()


## Step 19 — Prepare the modelling dataset

This section builds:
- **`X_model`** = the first 10 principal components
- **`y_model`** = the target variable `Diagnosis`

These are the inputs for classification modelling.


In [ ]:
X_model = df_pca.iloc[:, :num_pcs_to_use].copy()
y_model = df_pca["Diagnosis"].copy()

print("Preview of feature matrix X_model:")
display(X_model.head())

print("\nPreview of target vector y_model:")
display(y_model.head())


## Step 20 — Split the data into training and testing sets

A train-test split is used so that the models are evaluated on unseen data.

### Settings used
- test size = 20%
- random state = 42
- stratification = enabled where possible to preserve class balance


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42,
    stratify=y_model
)

print("Training and testing shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTarget distribution in y_train:")
display(y_train.value_counts())

print("\nTarget distribution in y_test:")
display(y_test.value_counts())


## Step 21 — Initialize multiple classification models

This notebook compares several common classifiers:
- Logistic Regression
- Support Vector Machine
- Decision Tree
- Random Forest
- K-Nearest Neighbors
- Naive Bayes
- Gradient Boosting

Using multiple models allows a more balanced comparison instead of relying on a single algorithm.


In [ ]:
classifiers = [
    ("Logistic Regression", LogisticRegression(random_state=42, max_iter=2000)),
    ("Support Vector Machine", SVC(random_state=42, probability=True)),
    ("Decision Tree", DecisionTreeClassifier(random_state=42)),
    ("Random Forest", RandomForestClassifier(random_state=42)),
    ("K-Nearest Neighbors", KNeighborsClassifier()),
    ("Naive Bayes", GaussianNB()),
    ("Gradient Boosting", GradientBoostingClassifier(random_state=42)),
]

print("Initialized models:")
for name, _ in classifiers:
    print("-", name)


## Step 22 — Train and evaluate all classification models

For each model, the notebook calculates:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC (where probability scores can be obtained)

The positive class is assumed to be **`M`** when the target contains labels such as `M` and `B`.


In [ ]:
performance_results = []
trained_models = {}

unique_classes = sorted(y_model.unique().tolist())
positive_label = "M" if "M" in unique_classes else unique_classes[-1]
binary_problem = len(unique_classes) == 2

for name, clf in classifiers:
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    trained_models[name] = clf

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=positive_label, zero_division=0) if binary_problem else np.nan,
        "Recall": recall_score(y_test, y_pred, pos_label=positive_label, zero_division=0) if binary_problem else np.nan,
        "F1 Score": f1_score(y_test, y_pred, pos_label=positive_label, zero_division=0) if binary_problem else np.nan,
    }

    roc_auc = np.nan
    if binary_problem:
        try:
            if hasattr(clf, "predict_proba"):
                y_score = clf.predict_proba(X_test)[:, 1]
                y_true_bin = (y_test == positive_label).astype(int)
                roc_auc = roc_auc_score(y_true_bin, y_score)
            elif hasattr(clf, "decision_function"):
                y_score = clf.decision_function(X_test)
                y_true_bin = (y_test == positive_label).astype(int)
                roc_auc = roc_auc_score(y_true_bin, y_score)
        except Exception:
            pass

    row["ROC-AUC"] = roc_auc
    performance_results.append(row)

results_df = pd.DataFrame(performance_results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
display(results_df)


## Step 23 — Present the model comparison table clearly

The following table ranks the models from best to worst using **F1 Score**.

This is usually a better ranking criterion than raw accuracy when both false positives and false negatives matter.


In [ ]:
results_styled = results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1 Score": "{:.4f}",
    "ROC-AUC": "{:.4f}",
}).background_gradient(subset=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"])

display(results_styled)


## Step 24 — Inspect the best-performing model in more detail

This section automatically selects the best model from the comparison table and shows:
- the confusion matrix
- the detailed classification report

This gives a more interpretable view of performance than a summary table alone.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_pred = best_model.predict(X_test)

print(f"Best model based on F1 Score: {best_model_name}\n")

print("Classification report:")
print(classification_report(y_test, best_pred))

cm = confusion_matrix(y_test, best_pred, labels=unique_classes)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=unique_classes,
    yticklabels=unique_classes
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()


## Step 25 — Use a linear SVM to inspect which principal components matter most

A linear SVM is helpful because its coefficients can indicate the relative contribution of each principal component to the classification boundary.


In [ ]:
linear_svm = SVC(kernel="linear", random_state=42)
linear_svm.fit(X_train, y_train)

coef_values = np.abs(linear_svm.coef_[0])

svm_importance_df = pd.DataFrame({
    "Principal Component": X_model.columns,
    "Absolute Coefficient": coef_values
}).sort_values(by="Absolute Coefficient", ascending=False)

print("Principal component importance from linear SVM:")
display(svm_importance_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=svm_importance_df, x="Principal Component", y="Absolute Coefficient")
plt.title("Importance of Principal Components from Linear SVM")
plt.xticks(rotation=45)
plt.show()


## Step 26 — Examine PCA loadings to connect PCs back to the original variables

PCA loadings show how strongly each original variable contributes to each principal component.

This is useful because classification is being done on PCs, but interpretation usually needs to return to the original variables.


In [ ]:
loadings = pca.components_
loadings_df = pd.DataFrame(
    loadings,
    columns=numeric_cols,
    index=[f"PC{i+1}" for i in range(loadings.shape[0])]
)

important_pcs = svm_importance_df["Principal Component"].head(min(3, len(svm_importance_df))).tolist()

print("Top contributing original variables for the most important PCs:")
for pc in important_pcs:
    print(f"\nTop loadings for {pc}:")
    display(loadings_df.loc[pc].sort_values(key=np.abs, ascending=False).head(10).to_frame(name="Loading"))

pc_to_plot = important_pcs[0]
plt.figure(figsize=(12, 6))
loadings_df.loc[pc_to_plot].sort_values(key=np.abs, ascending=False).head(15).plot(kind="bar")
plt.title(f"Top original variable loadings for {pc_to_plot}")
plt.ylabel("Loading value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Step 27 — Final analytical summary

At this point, the notebook has:
1. loaded user-provided data
2. checked the dataset structure
3. removed an identifier field if present
4. explored distributions and outliers
5. cleaned the data
6. applied PCA after scaling
7. trained multiple classification models
8. compared their performance
9. interpreted the best model and the most important principal components

### Suggested next improvements
- add cross-validation for more robust model evaluation
- perform hyperparameter tuning for the best models
- compare PCA-based models with models trained on original features
- use SHAP or permutation importance for more detailed interpretation
- save outputs automatically to Excel or CSV for reporting
